<a href="https://colab.research.google.com/github/YaminisriVellanki/24B11DS221_DriveReady_OPC_2026/blob/main/task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 – Prompts, Memory and Python for AI Agents

**Objective:** Learn prompting, structured JSON outputs, conversation memory and essential Python concepts to build a chatbot.

**Case Study:** Sweet Crumbs Home Bakery

**Practical Goal:** Build a chatbot that understands customer messages, remembers conversations, processes orders and handles errors.

In [1]:
!pip install -q openai

In [2]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("Agentic_ai_api_key")

client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://api.groq.com/openai/v1"
)

print("API client configured successfully")

API client configured successfully


In [3]:
MODEL = "openai/gpt-oss-20b"

### Task 1: Prompt Parts

First, remove the RULES from the system prompt and ask whether the bakery can deliver in one hour. Then restore the RULES and ask the same question. Compare the responses.

In [4]:
SYSTEM_WITHOUT_RULES = """ROLE: You are the assistant for Sweet Crumbs Home Bakery.
FACTS: Orders need 24 hours' notice. All cakes have eggless versions.
       Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

messages = [
    {"role": "system", "content": SYSTEM_WITHOUT_RULES},
    {"role": "user", "content": "Can you deliver in 1 hour?"}
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0.3
)

print("Without RULES:")
print(response.choices[0].message.content)

Without RULES:
Hi! Unfortunately we need 24 hours’ notice for orders. All cakes are eggless. Let us know what you’d like and we’ll schedule it. Thanks!


In [5]:
SYSTEM_WITH_RULES = """ROLE: You are the assistant for Sweet Crumbs Home Bakery.
FACTS: Orders need 24 hours' notice. All cakes have eggless versions.
       Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
RULES: Never promise a delivery time. If the answer isn't in FACTS,
       say "I'll check with the owner."
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

messages = [
    {"role": "system", "content": SYSTEM_WITH_RULES},
    {"role": "user", "content": "Can you deliver in 1 hour?"}
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0.3
)

print("With RULES:")
print(response.choices[0].message.content)

With RULES:
We need 24 hours’ notice for orders, so we can’t deliver in 1 hour. Let me know if you’d like to place an order for a later time.


### Task 2: Few-shot Classification for a Business

In this task, I am creating a customer-message classifier using few-shot prompting.

I provide three example messages with their correct labels:
- order
- question
- complaint

Then I test the classifier with five new customer messages.

Objective: Understand how examples guide the model to produce consistent classifications.

In [6]:
examples = [
    {"role": "user", "content": "need 2 plum cakes for friday"},
    {"role": "assistant", "content": "order"},

    {"role": "user", "content": "do you have eggless?"},
    {"role": "assistant", "content": "question"},

    {"role": "user", "content": "the cake was dry yesterday"},
    {"role": "assistant", "content": "complaint"}
]

test_messages = [
    "I want 5 cupcakes tomorrow",
    "Are chocolate cakes available?",
    "The cake was damaged",
    "Can I order an eggless plum cake?",
    "Your delivery was late"
]

for message in test_messages:
    messages = [
        {
            "role": "system",
            "content": "Classify the customer message using exactly one word: order, question or complaint."
        }
    ] + examples + [
        {"role": "user", "content": message}
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2
    )

    print("Customer:", message)
    print("Classification:", response.choices[0].message.content)
    print("-" * 50)

Customer: I want 5 cupcakes tomorrow
Classification: order
--------------------------------------------------
Customer: Are chocolate cakes available?
Classification: question
--------------------------------------------------
Customer: The cake was damaged
Classification: complaint
--------------------------------------------------
Customer: Can I order an eggless plum cake?
Classification: question
--------------------------------------------------
Customer: Your delivery was late
Classification: complaint
--------------------------------------------------


### Task 3: Extracting Customer Orders as JSON

In this task, I am asking the model to extract order information from a normal customer message and return it in JSON format.

The required fields are name, item, qty, date and eggless.

I use json.loads() to convert the model's JSON response into a Python dictionary. I use try-except to handle invalid JSON responses.

Objective: Convert unstructured customer messages into structured data that Python can process.

In [7]:
import json

SYSTEM_JSON = """Extract the order from the customer's message.
Return ONLY JSON, no other text, with these keys:
name (text), item (text), qty (number), date (text), eggless (true/false)."""

messages = [
    {"role": "system", "content": SYSTEM_JSON},
    {
        "role": "user",
        "content": "Hi Ravi here, need 2 plum cakes for 24th December, eggless please."
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0.2
)

reply = response.choices[0].message.content

print("Model response:")
print(reply)

try:
    order = json.loads(reply)

    print("\nExtracted Order:")
    print("Name:", order["name"])
    print("Item:", order["item"])
    print("Quantity:", order["qty"])
    print("Date:", order["date"])
    print("Eggless:", order["eggless"])

except json.JSONDecodeError as e:
    print("JSON parsing error:", e)

Model response:
{"name":"Ravi","item":"plum cakes","qty":2,"date":"24th December","eggless":true}

Extracted Order:
Name: Ravi
Item: plum cakes
Quantity: 2
Date: 24th December
Eggless: True


### Task 4: Testing Chatbot Memory

In this task, I am reducing MAX_TURNS to 2 and testing the chatbot's memory.

First, I tell the chatbot my name. Then I have three more conversation turns and ask it to recall my name.

Objective: Understand how trimming conversation history affects what the model can remember.

In [9]:
import json

MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

SYSTEM = f"""
You are a friendly assistant for Sweet Crumbs Home Bakery.

MENU:
{json.dumps(MENU)}

FACTS:
- Orders require 24 hours notice.
- Eggless cakes are available.

RULES:
- Never promise a delivery time.
- If you do not know something, ask the owner.
- Remember the customer's name and order.

FORMAT:
- Keep responses under 50 words.
- Reply in English or Telugu.
"""

In [10]:
history = [
    {"role": "system", "content": SYSTEM}
]

MAX_TURNS = 2

print("Memory setup completed!")

Memory setup completed!


In [11]:
{"role": "system", "content": SYSTEM}

{'role': 'system',
 'content': '\nYou are a friendly assistant for Sweet Crumbs Home Bakery.\n\nMENU:\n{"plum cake": 500, "chocolate cake": 650, "cupcake": 80}\n\nFACTS:\n- Orders require 24 hours notice.\n- Eggless cakes are available.\n\nRULES:\n- Never promise a delivery time.\n- If you do not know something, ask the owner.\n- Remember the customer\'s name and order.\n\nFORMAT:\n- Keep responses under 50 words.\n- Reply in English or Telugu.\n'}

In [12]:
def chat_with_limited_memory(user_text: str) -> str:
    history.append({
        "role": "user",
        "content": user_text
    })

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    response = client.chat.completions.create(
        model=MODEL,
        messages=recent,
        temperature=0.3
    )

    reply = response.choices[0].message.content

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

In [13]:
print("Turn 1:", chat_with_limited_memory(
    "My name is Yamini."
))

print("Turn 2:", chat_with_limited_memory(
    "I like chocolate cake."
))

print("Turn 3:", chat_with_limited_memory(
    "What cakes do you sell?"
))

print("Memory Test:", chat_with_limited_memory(
    "What is my name?"
))

Turn 1: Hello Yamini! 👋 How can I help you today? Are you interested in ordering a plum cake, chocolate cake, or a cupcake? Let me know your choice and any special requests.
Turn 2: Great choice, Yamini! 🍫 Our chocolate cake is 650. Remember, we need 24‑hour notice for orders. Would you like an eggless version or any special decorations? Let me know!
Turn 3: We sell plum cake (₹500), chocolate cake (₹650), and cupcakes (₹80). All are eggless. Orders need 24‑hour notice. Let me know which you’d like!
Memory Test: I’m sorry, I don’t have that info. Could you tell me your name?


## Task 5: Build a Chatbot for My Own Business – DemandPulse AI

In this task, I create a chatbot for DemandPulse AI, my demand forecasting solution.

I use a Python dictionary to store the available services, a function with type hints and a docstring, and a custom SYSTEM prompt.

The objective is to apply Day 2 concepts to my own AI-based business example.

In [14]:
BUSINESS_CATALOGUE = {
    "sales forecasting": 500,
    "demand analysis": 300,
    "inventory planning": 400,
    "business report": 200
}

print("DemandPulse AI Services:")
print(BUSINESS_CATALOGUE)

DemandPulse AI Services:
{'sales forecasting': 500, 'demand analysis': 300, 'inventory planning': 400, 'business report': 200}


### Price Calculation Function

I define a function to calculate the total service cost based on the selected service and quantity.

I use type hints and a docstring to make the function clear and reusable.

In [15]:
def get_service_price(service: str, qty: int = 1) -> int:
    """
    Calculate the total price of a DemandPulse AI service.

    Args:
        service: Name of the selected service.
        qty: Number of service units.

    Returns:
        Total service price.
    """
    price = BUSINESS_CATALOGUE.get(service.lower())

    if price is None:
        return 0

    return price * qty


print("Total Price:", get_service_price("sales forecasting", 2))

Total Price: 1000


### Custom SYSTEM Prompt

I create a SYSTEM prompt for DemandPulse AI.

It defines the assistant's role, available services, business information and response rules.

In [16]:
SYSTEM_BUSINESS = f"""
You are a helpful AI assistant for DemandPulse AI.

SERVICES:
{json.dumps(BUSINESS_CATALOGUE)}

FACTS:
- DemandPulse AI helps businesses understand demand patterns.
- It provides sales forecasting and demand analysis services.
- It supports inventory planning and business reporting.

RULES:
- Answer using the available service catalogue.
- Never invent service prices.
- If a service is unavailable, ask the customer to contact the team.
- Keep responses short and clear.
"""
print("DemandPulse AI SYSTEM prompt created successfully!")

DemandPulse AI SYSTEM prompt created successfully!


### Testing the DemandPulse AI Chatbot

I send a customer query to the chatbot using the custom SYSTEM prompt.

The objective is to check whether the chatbot correctly identifies services and provides catalogue-based information.

In [17]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_BUSINESS},
        {"role": "user", "content": "What services does DemandPulse AI provide?"}
    ],
    temperature=0.3
)

print(response.choices[0].message.content)

DemandPulse AI offers the following services:

- **Sales forecasting** – $500  
- **Demand analysis** – $300  
- **Inventory planning** – $400  
- **Business report** – $200


In [18]:
print("Sales Forecasting:", get_service_price("sales forecasting"))
print("Demand Analysis:", get_service_price("demand analysis"))
print("Two Inventory Planning units:", get_service_price("inventory planning", 2))
print("Unknown Service:", get_service_price("customer support"))

Sales Forecasting: 500
Demand Analysis: 300
Two Inventory Planning units: 800
Unknown Service: 0
